# CP2 · Notebook 06 — Failure modes

## Objetivo

Inspeccionar específicamente las imágenes del `cp2-depth-extras/` (challenges) y ver dónde ambos modelos mienten. **No** es para que las arregles — es para que las identifiques.

**Tiempo estimado**: 10 min.

## 1. Cargar depth maps de challenge

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

OUT_DIR = Path('../outputs')
CP2_EXTRAS = Path('../datasets/cp2-depth-extras')

with open(OUT_DIR / '03_midas_summary.json') as f:
    midas_summary = json.load(f)
with open(OUT_DIR / '05_da_summary.json') as f:
    da_summary = json.load(f)

challenges_m = [e for e in midas_summary['per_image'] if e['category'] == 'challenge']
challenges_d = [e for e in da_summary['per_image']    if e['category'] == 'challenge']
print(f'Challenges: {len(challenges_m)} (MiDaS) · {len(challenges_d)} (DA v2)')
for e in challenges_m:
    print(f'  - {e["name"]}')

## 2. Visualizar uno a uno con anotación de qué buscamos

Para cada imagen `challenge`, mostramos:
- Original
- MiDaS depth
- DA v2 depth

Y comentamos lo que **deberías observar** en cada caso.

In [ ]:
expectations = {
    'reflection': 'Superficie reflectante (cristal, agua, carrocería pulida). Esperado: el modelo "ve" la profundidad del reflejado, no la superficie real.',
    'sky':        'Cielo plano sin textura. Esperado: profundidad inconsistente o aleatoria en la zona cielo.',
    'tunnel':     'Túnel con iluminación pobre. Esperado: depth ruidoso, posibles "saltos" en zonas oscuras.',
    'night':      'Escena nocturna con luces puntuales. Esperado: el modelo concentra atención en farolas, depth incorrecto en zonas oscuras.',
    'transparent':'Objeto semitransparente (cristal, plástico). Esperado: depth shift, el modelo "ve a través".',
    'low_texture':'Zona sin texturas (pared blanca, asfalto homogéneo). Esperado: depth aplanado, sin gradiente.',
    'default':    'Imagen "trampa" — observa qué hace el modelo en zonas inusuales.',
}

def expected_for(name):
    low = name.lower()
    for key in expectations:
        if key in low: return expectations[key]
    return expectations['default']

MIDAS_DIR = OUT_DIR / '03_midas_depth'
DA_DIR    = OUT_DIR / '05_da_depth'

rows = []
for e_m, e_d in zip(challenges_m, challenges_d):
    name = e_m['name']
    img = np.array(Image.open(CP2_EXTRAS / name).convert('RGB'))
    midas_d = np.load(MIDAS_DIR / e_m['npy_file'])
    da_d    = np.load(DA_DIR    / e_d['npy_file'])
    rows.append((name, img, midas_d, da_d, expected_for(name)))

for name, img, midas_d, da_d, expected in rows:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(img); axes[0].set_title(f'Original · {name}'); axes[0].axis('off')
    im1 = axes[1].imshow(midas_d, cmap='plasma'); axes[1].set_title('MiDaS depth'); axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
    im2 = axes[2].imshow(da_d, cmap='plasma'); axes[2].set_title('DA v2 depth'); axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046)
    plt.tight_layout(); plt.show()
    print(f'📝 {name}:  {expected}\n')

## 3. Análisis del cielo (alta varianza intra-clase)

Heurística: una zona realmente "a infinito" debería tener inverse_depth ≈ 0 con baja varianza. Si la varianza es alta el modelo está **inventando** estructura.

Calculamos la varianza del depth en la mitad superior de cada imagen (asumiendo que ahí hay más cielo).

In [ ]:
print(f"{'imagen':<25s} {'MiDaS top-var':>15s} {'DA v2 top-var':>15s}")
print('-' * 60)
for name, img, midas_d, da_d, _ in rows:
    h = midas_d.shape[0]
    midas_top = midas_d[:h//3].std() / (midas_d.std() + 1e-6)
    h2 = da_d.shape[0]
    da_top = da_d[:h2//3].std() / (da_d.std() + 1e-6)
    print(f'{name:<25s}  {midas_top:>15.3f}  {da_top:>15.3f}')

print('\nLa columna "top-var" muestra varianza relativa en el tercio superior. Valores >0.5')
print('suelen indicar que el modelo está inventando estructura donde no hay (cielo, lente sucio).')

## 4. Tu turno — probar imagen propia

Si tienes una foto de tu calle o un dashcam propio, descomprime este bloque y prueba. Es el **extra opcional** que más puntos suma en la rúbrica.

In [ ]:
# Descomenta y cambia la ruta:
# from transformers import AutoImageProcessor, AutoModelForDepthEstimation
# import torch
#
# OWN_IMAGE = Path('~/Downloads/mi-foto.jpg').expanduser()  # ← tu imagen
# img = Image.open(OWN_IMAGE).convert('RGB')
#
# proc_m = AutoImageProcessor.from_pretrained('Intel/dpt-swinv2-tiny-256')
# mod_m  = AutoModelForDepthEstimation.from_pretrained('Intel/dpt-swinv2-tiny-256').eval()
# proc_d = AutoImageProcessor.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf')
# mod_d  = AutoModelForDepthEstimation.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf').eval()
#
# with torch.no_grad():
#     m_out = mod_m(**proc_m(images=img, return_tensors='pt')).predicted_depth[0].numpy()
#     d_out = mod_d(**proc_d(images=img, return_tensors='pt')).predicted_depth[0].numpy()
#
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# axes[0].imshow(img); axes[0].set_title(OWN_IMAGE.name); axes[0].axis('off')
# axes[1].imshow(m_out, cmap='plasma'); axes[1].set_title('MiDaS'); axes[1].axis('off')
# axes[2].imshow(d_out, cmap='plasma'); axes[2].set_title('DA v2'); axes[2].axis('off')
# plt.show()
print('Bloque comentado — descomenta + ajusta la ruta para probar tu imagen.')

## 5. Cierre

Has visto en qué tipo de imagen los foundation models **mienten**. Los failure modes recurrentes:

- **Reflejos** → el modelo ve la escena reflejada, no la superficie.
- **Cielo / sin textura** → inventa estructura aleatoria.
- **Iluminación adversa** (noche, túnel) → ruido extremo.
- **Transparencias** → "ve a través".

**Para `07_preguntas.md`**: identifica al menos 1 failure mode adicional propio (sobre una imagen `easy/medium/hard` o tuya) y razónalo.

Ve a `07_preguntas.md` para entregar.